# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akhilabusalih/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# FlyRank Content Decay: A Human-Reviewed Refresh Prioritization Study

## Abstract

This paper asks whether observed search, engagement, and content signals can help rank pages for refresh review. It uses FlyRank's public-safe 30,000-row starter export, while documenting that the larger warehouse release exists but is not the source of these results. A leakage-aware feature set and random forest are evaluated against a transparent rule baseline on a client-grouped holdout. On this snapshot, the model measured average precision of 0.610 and Precision@50 of 0.680, compared with 0.468 and 0.240 for the baseline. The result supports directional, human-reviewed prioritization, not a causal claim that refreshing a page will improve traffic, rankings, or revenue.

## Introduction / Problem statement

Content teams need to decide which page to inspect first when many pages show signs of decline. A fixed rule can be useful, but a learned ranking may combine more signals while preserving a human decision at the end. This study therefore treats the model as a review queue: it ranks pages for attention, rather than automatically changing or publishing content.

The central question is deliberately narrow: **does the measured model ranking provide useful decision support beyond the transparent baseline on a client-grouped holdout?**

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "scripts").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not locate repository root")


ROOT = find_repo_root(Path.cwd())
WORK_OUTPUT = ROOT / "work" / "outputs"
WORK_FIGURES = ROOT / "work" / "figures"
metrics = json.loads((WORK_OUTPUT / "action_playbook_metrics.json").read_text(encoding="utf-8"))
model_results = json.loads((WORK_OUTPUT / "model_results.json").read_text(encoding="utf-8"))
playbook_summary = json.loads((WORK_OUTPUT / "action_playbook_summary.json").read_text(encoding="utf-8"))

print("Research question: can a leakage-aware model rank observed decline-risk pages for human review?")
print(f"Rows scored: {metrics['rows_scored']:,}")
print(f"Observed positive-label rate: {metrics['target_positive_rate']:.1%}")

Research question: can a leakage-aware model rank observed decline-risk pages for human review?
Rows scored: 30,000
Observed positive-label rate: 54.2%


## 2. Data

The reproducible analysis uses `data/raw/content_refresh_anonymized.csv`: 30,000 pseudonymized content-item rows with 90-day search and engagement summaries and content metadata. The broader FlyRank warehouse release contains daily performance, content, client, and query tables, but this paper does **not** claim to have trained on its approximately 79 million daily rows. The starter export is the public-safe source behind every number shown here.

The feature windows are the supplied trailing 90-day summaries, with recent and previous 30-day fields where available. The target is `is_declining_label = (trend_direction == "down")`. IDs are used only for joins and client grouping. `trend_direction` and `trend_pct` are excluded because they define or directly encode the label; product flags and scores are also excluded as decision-derived signals.

Rates such as CTR are read in the dataset's percentage-point convention, and `avg_position = 0` is treated as no data rather than rank zero. No client names, domains, URLs, titles, or raw queries are included in the paper.

In [2]:
question_frame = pd.DataFrame([
    {
        "decision": "which content item should a reviewer inspect first?",
        "unit": "one pseudonymous content item",
        "output": "rank, confidence, suggested action, and reason codes",
        "automation_boundary": "human approval remains required",
    }
])
display(question_frame)

,decision,unit,output,automation_boundary
0,which content item should a reviewer inspect f...,one pseudonymous content item,"rank, confidence, suggested action, and reason...",human approval remains required


## 3. Methodology

The target is binary classification of the observed decline proxy. Numeric features include demand, activity, age, position, CTR, engagement, scroll, AI traffic, and log-transformed volume fields; categorical features include content type, intent, age, freshness, word-count, impression, and position tiers.

The baseline is a transparent rules score. The model comparison includes logistic regression, a shallow decision tree, and a random forest; the random forest is selected by Precision@50. The recorded validation split is client-aware: rows from held-out clients are kept out of training. The positive-label base rate is reported beside performance so accuracy is not read without context.

Leakage checks exclude label-derived fields (`trend_direction`, `trend_pct`), IDs (`content_id`, `client_id`), and product decision flags. The flat starter export does not prove every feature timestamp precedes the label window, so temporal strictness remains a limitation rather than an unstated assumption.

In [3]:
model_table = pd.DataFrame([
    {
        "method": name,
        "ROC AUC": values["roc_auc"],
        "Average precision": values["average_precision"],
        "Precision@50": values["precision_at_50"],
        "Recall": values["recall"],
        "F1": values["f1"],
    }
    for name, values in model_results["models"].items()
] + [{
    "method": "baseline_rules",
    "ROC AUC": model_results["baseline"]["baseline_roc_auc"],
    "Average precision": model_results["baseline"]["baseline_average_precision"],
    "Precision@50": model_results["baseline"]["baseline_precision_at_50"],
    "Recall": model_results["baseline"]["baseline_recall"],
    "F1": model_results["baseline"]["baseline_f1"],
}])

for column in ["ROC AUC", "Average precision", "Precision@50", "Recall", "F1"]:
    model_table[column] = model_table[column].round(3)

print("Measured model and baseline receipt:")
display(model_table)
print(f"Validation split: {model_results['split_strategy']}; base rate: {model_results['target_positive_rate']:.1%}")

Measured model and baseline receipt:


,method,ROC AUC,Average precision,Precision@50,Recall,F1
0,decision_tree,0.742,0.575,0.58,0.716,0.634
1,logistic_regression,0.700,0.522,0.40,0.567,0.566
2,random_forest,0.747,0.610,0.68,0.741,0.638
3,baseline_rules,0.627,0.468,0.24,0.189,0.274


Validation split: client_holdout; base rate: 54.2%


## 4. Results (model vs baseline)

On the same client-grouped holdout, the random forest measured AP 0.610 and Precision@50 0.680. The rule baseline measured AP 0.468 and Precision@50 0.240. These are measured ranking comparisons on this snapshot, not evidence that a refresh caused an outcome.

The observed positive-label rate is 54.2%, so accuracy is not the primary decision metric. Precision@50 is shown because the practical decision is which small set of pages a reviewer should inspect first.

In [4]:
chart_paths = [
    WORK_FIGURES / "action_mix.svg",
    WORK_FIGURES / "archetype_mix.svg",
]
chart_status = pd.DataFrame({
    "chart": [path.name for path in chart_paths],
    "exists": [path.exists() and path.stat().st_size > 0 for path in chart_paths],
})
display(chart_status)
assert chart_status["exists"].all()

print("Paper chart captions:")
print("- Action mix: the queue is a review prioritization, with monitor kept separate from refresh actions.")
print("- Archetype mix: visible low-CTR and visible-decline patterns are reason-code groupings, not causal categories.")

,chart,exists
0,action_mix.svg,True
1,archetype_mix.svg,True


Paper chart captions:
- Action mix: the queue is a review prioritization, with monitor kept separate from refresh actions.
- Archetype mix: visible low-CTR and visible-decline patterns are reason-code groupings, not causal categories.


## 5. Limitations & honest framing

This is an observational ranking study on one public-safe snapshot. The label is a trend-derived proxy, not a direct measurement of business value; the model has not been tested on a future time period in this notebook; and the flat starter export cannot fully prove feature-window timing. Client grouping improves the validation question for unseen clients, but it does not establish generalization to every future client.

The paper does not claim that the model predicts Google's algorithm, that any feature causes decline, or that refreshing a page will improve traffic, rankings, or revenue. The queue is decision-support: a human should inspect the evidence, page context, factual accuracy, and editorial constraints before acting.

In [5]:
limitations = pd.DataFrame([
    ["Snapshot", "Results describe this 30,000-row release and may shift with new data."],
    ["Label", "The decline label is a trend-derived proxy, not a causal or revenue outcome."],
    ["Validation", "Client grouping supports an unseen-client question; future-time performance remains untested here."],
    ["Action", "The queue prioritizes review; it does not automate publishing or prove refresh impact."],
], columns=["limit", "honest interpretation"])
display(limitations)

,limit,honest interpretation
0,Snapshot,"Results describe this 30,000-row release and m..."
1,Label,"The decline label is a trend-derived proxy, no..."
2,Validation,Client grouping supports an unseen-client ques...
3,Action,The queue prioritizes review; it does not auto...


## 6. Ranked recommendations

The action playbook turns the ranking into a practical review sequence:

1. Start with high-confidence, high-score items that have visible demand and `refresh_and_review_ctr` or `refresh` reason codes.
2. Review visible low-CTR pages for title and snippet alignment before changing copy.
3. Review engagement and scroll signals for usefulness, readability, and navigation problems.
4. Use `expand_and_refresh` only as a prompt to inspect coverage gaps, not as permission to add unsupported text.
5. Keep lower-confidence items in monitoring batches and revisit them when the next measured snapshot arrives.

No recommendation should be executed automatically. The reviewer records accept, defer, or reject and keeps the reason for that decision.

In [6]:
action_counts = pd.Series(metrics["action_counts"], name="rows").sort_values(ascending=False)
recommendation_receipt = pd.DataFrame({
    "action": action_counts.index,
    "rows": action_counts.values,
    "share": (action_counts.values / metrics["rows_scored"]).round(3),
})
display(recommendation_receipt)
print(f"Highest-ranked first action: {playbook_summary['top_ranked_action']}")
print("Use boundary: human-reviewed prioritization only; no automatic publishing, deletion, or causal interpretation.")

,action,rows,share
0,monitor,13069,0.436
1,refresh,8207,0.274
2,refresh_and_review_ctr,6655,0.222
3,refresh_and_review_engagement,1987,0.066
4,expand_and_refresh,82,0.003


Highest-ranked first action: refresh_and_review_ctr
Use boundary: human-reviewed prioritization only; no automatic publishing, deletion, or causal interpretation.


## 7. Reproducibility

The repository contains the complete workflow and receipts:

- [Week 5 model notebook](w05_model.ipynb)
- [Week 6 validation audit](w06_validation_audit.ipynb)
- [Week 7 action playbook](w07_action_playbook.ipynb)
- [Feature preparation script](../../scripts/01_prepare_features.py)
- [Model training script](../../scripts/03_train_model.py)
- [Action export script](../../scripts/04_evaluate_and_export.py)
- [Committed model receipt](../outputs/model_results.json)
- [Committed action-playbook receipt](../outputs/action_playbook_metrics.json)

From a fresh environment, install `requirements.txt`, run the preparation, baseline, training, and export scripts, then run the notebooks top to bottom. The pipeline uses fixed random state 42 where model training and client holdout selection require randomness. The deployed page is generated from the same committed receipts and figures.

In [7]:
required_paths = [
    WORK_OUTPUT / "model_results.json",
    WORK_OUTPUT / "action_playbook_metrics.json",
    WORK_OUTPUT / "action_playbook_summary.json",
    WORK_FIGURES / "action_mix.svg",
    WORK_FIGURES / "archetype_mix.svg",
]
reproducibility = pd.DataFrame({
    "artifact": [str(path.relative_to(ROOT)) for path in required_paths],
    "present": [path.exists() and path.stat().st_size > 0 for path in required_paths],
})
display(reproducibility)
assert reproducibility["present"].all()
print("Reproducibility receipt checks passed.")

,artifact,present
0,work/outputs/model_results.json,True
1,work/outputs/action_playbook_metrics.json,True
2,work/outputs/action_playbook_summary.json,True
3,work/figures/action_mix.svg,True
4,work/figures/archetype_mix.svg,True


Reproducibility receipt checks passed.


## 8. Acknowledgments & data credit

Built on the [FlyRank ML Internship dataset](https://flyrank.ai/). The release is public-safe and pseudonymized; this paper does not disclose client names, domains, URLs, or private queries. Thanks to the FlyRank team for making a careful, reproducible modeling exercise possible.

## Self-check

- [x] Title and five-sentence abstract are at the top.
- [x] Introduction, data, methodology, results, limitations, recommendations, reproducibility, and data credit are present.
- [x] Model and baseline are compared on the same recorded holdout with the base rate visible.
- [x] Charts and receipts are linked through relative repository paths.
- [x] Claims use measured, observed, directional, and decision-support language.
- [x] No causal refresh claim, client-identifying detail, raw query, or private URL is included.